# ArrilesLLM-Cosplay | 多语言角色扮演模型微调指南

**Google Colab Pro · Gemma 4 E2B · QLoRA · SwanLab · GGUF 导出**

**文档版本：2.1（10,000 条合成候选数据）**  
**适用对象：** 使用 Google Colab 开展监督微调（SFT）的开发者及研究人员  
**模型标识：** ArrilesLLM-Cosplay（ArrilesLLM 系列）  
**开发及训练主体：** Arriles 项目组  
**基础模型：** Google Gemma 4 E2B（`unsloth/gemma-4-E2B-it`）

## 文档概述

本文档提供基于 Gemma 4 E2B 的多语言角色扮演模型微调流程，覆盖 GPU 环境配置、数据集导入与校验、模型身份监督数据构建、QLoRA 参数高效微调、SwanLab 实验跟踪、推理验证以及模型导出。项目面向**通用多角色扮演场景**，训练内容涵盖日常交流和叙事互动两类任务。

### 数据与训练范围

- **角色扮演数据：** 10,000 段合成对话，覆盖中文、英语、日语、韩语、西班牙语，各 400 段。
- **模型身份数据：** 100 段多语言问答，分别加入训练、验证和测试数据。
- **数据划分：** 原始角色扮演数据按训练集 1,600 段、验证集 200 段、测试集 200 段读取，不在 Notebook 中重新划分。
- **监督目标：** 使用多轮对话的 `prompt`/`completion` 结构，仅对目标回复计算损失。
- **实验跟踪：** 通过 `SwanLabCallback` 记录训练与评估指标，并将检查点及 Adapter 保存至 Google Drive。

### 执行说明

请按各章节顺序执行代码单元。首次运行使用 `SMOKE_TEST=True` 完成 20 步功能验证；确认资源、训练日志与模型输出正常后，再启动正式训练。该 Notebook 仅用于**微调已有基础模型**，不涉及从随机权重预训练。

> **验证状态与数据限制：** 本文件已执行本地语法与数据结构检查，尚未在 Colab GPU 上完成端到端训练。配套数据包含模板生成内容，适用于功能联调和初步实验；其样本数量不等同于 10,000 段相互独立的高质量原创剧情。实际训练效果须以独立测试集评估为准。

### 参考文档

[Gemma / Unsloth 微调指南](https://gemma.ai/tuning/unsloth/) · [SwanLab TRL 集成](https://docs.swanlab.cn/guide_cloud/integration/integration-huggingface-trl.html) · [Hugging Face TRL SFT](https://huggingface.co/docs/trl/sft_trainer)


## 1. 运行环境配置

在 Google Colab 中打开“代码执行程序 → 更改运行时类型”，选择 **Python 3** 和可用的 **L4 或 A100 GPU**。T4 可用于资源受限的兼容性测试，但可能需要进一步缩减训练配置。

执行下方代码核验 PyTorch、CUDA 状态、GPU 型号及显存容量。Colab 运行时可能因配额或会话限制而中断；持久化文件应保存至 Google Drive。

In [ ]:
import os, subprocess, torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('未检测到 GPU。请在 Colab 中切换 GPU 运行时。')
print('GPU:', torch.cuda.get_device_name(0))
print('显存 GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
subprocess.run(['nvidia-smi'], check=False)

## 2. 安装训练依赖

本项目使用 Unsloth、Transformers、TRL、PEFT、Datasets、Accelerate、SentencePiece 及 SwanLab。

执行安装单元后，如 Colab 提示重启运行时，应按提示重启，并从环境检测单元重新执行。依赖接口可能随版本变化；如出现兼容性错误，应结合安装日志检查具体版本。

In [ ]:
%pip install -q -U unsloth "transformers>=5.10.1" "trl>=0.16.0" "peft>=0.19.0" datasets accelerate sentencepiece swanlab

## 3. 配置 Google Drive 存储

将 Google Drive 挂载为训练输出的持久化存储位置，保存实验检查点、LoRA Adapter、处理器配置和训练元数据。Notebook 默认将文件写入 `MyDrive/gemma4_roleplay`。

Google Drive 授权由 Colab 发起。未持久化至 Drive 的 `/content` 文件可能在运行时结束后丢失。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/gemma4_roleplay'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(PROJECT_DIR)

## 4. 导入多语言角色扮演数据集

本项目使用配套数据文件 `ArrilesLLM-Cosplay_V2_多语言角色扮演数据集_10000条.zip`。可通过 Colab 文件面板上传至 `/content/`，也可在运行下方单元时选择 ZIP 文件。

程序直接从压缩包读取 `train.jsonl`、`validation.jsonl`、`test.jsonl`，保留原数据划分，不执行二次随机切分，以降低关联场景跨数据集泄漏的风险。

**预期数据规模：** 训练集 1,600 段、验证集 200 段、测试集 200 段。
**数据来源与限制：**本数据集为多语言模板组合生成的合成训练候选集，存在较明显的重复回复和模板共享，不应直接视为已达到生产质量的角色扮演语料。


In [ ]:
import json
import zipfile
from pathlib import Path

ZIP_PATH = Path('/content/ArrilesLLM-Cosplay_V2_多语言角色扮演数据集_10000条.zip')
if not ZIP_PATH.exists():
    from google.colab import files
    print('请上传配套的 10000 条角色扮演数据 ZIP。')
    uploads = files.upload()
    matched = [Path('/content') / name for name in uploads if name.lower().endswith('.zip')]
    if len(matched) != 1:
        raise ValueError('请只上传一个 ZIP 数据集文件。')
    ZIP_PATH = matched[0]

# 仅从压缩包读取指定数据分割文件，避免解压非预期路径。
def load_split_from_zip(zip_file, filename):
    with zip_file.open(filename, 'r') as fp:
        return [json.loads(line.decode('utf-8')) for line in fp if line.strip()]

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    names = set(zf.namelist())
    required = {'train.jsonl', 'validation.jsonl', 'test.jsonl'}
    if not required.issubset(names):
        raise ValueError(f'ZIP 缺少文件: {required - names}；实际文件: {sorted(names)}')
    train_raw = load_split_from_zip(zf, 'train.jsonl')
    val_raw = load_split_from_zip(zf, 'validation.jsonl')
    test_raw = load_split_from_zip(zf, 'test.jsonl')

assert (len(train_raw), len(val_raw), len(test_raw)) == (8000, 1000, 1000), '数据集划分数量不符合配套 ZIP 预期'
print(f'原始对话：训练 {len(train_raw)} | 验证 {len(val_raw)} | 测试 {len(test_raw)}')
print('语言与角色示例：', train_raw[0]['language'], train_raw[0]['role_card']['name'])

## 5. 模型身份监督样本

V2 数据集中已经包含多语言模型身份问答（`genre=model_identity`，请注意实际分类字段为 `identity`）。**不要再次追加旧版 100 条身份样本**，以免加重模板重复。模型真实身份为 ArrilesLLM-Cosplay（ArrilesLLM 系列），由 Arriles 项目组基于 Google Gemma 4 E2B 训练。


In [ ]:
# V2 数据中已内置模型身份问答，此处只声明元数据，不额外生成样本。
MODEL_IDENTITY = {
    'name': 'ArrilesLLM-Cosplay',
    'family': 'ArrilesLLM',
    'team': 'Arriles 项目组',
    'base': 'Google Gemma 4 E2B',
}
print('原始记录数:', len(train_raw), len(val_raw), len(test_raw))
assert (len(train_raw), len(val_raw), len(test_raw)) == (8000, 1000, 1000)


## 6. 数据校验与监督样本构建

本节验证数据记录的基本结构、消息顺序和文本内容，并将原始完整对话转换为 TRL 接受的 conversational `prompt`/`completion` 格式。

原始角色设定使用 `system` 消息提供。为适配当前 Notebook 的 Gemma 4 对话处理方式，程序将该设定合并至首条 `user` 消息，并在后续轮次保留对话历史。每条 `assistant` 回复对应一个独立监督目标。

训练配置启用 `completion_only_loss=True`，仅对回复部分计算损失；角色卡和用户输入作为上下文。原始训练、验证和测试划分保持不变。

In [ ]:
import hashlib
from collections import Counter
from datasets import Dataset

def validate_record(row):
    assert isinstance(row.get('id'), str) and row['id'], '缺少 id'
    assert isinstance(row.get('role_card'), dict), f"{row['id']}: 缺少 role_card"
    messages = row.get('messages')
    assert isinstance(messages, list) and len(messages) >= 3, f"{row['id']}: 消息数不足"
    assert messages[0]['role'] == 'system', f"{row['id']}: 第一条必须是 system"
    rest = messages[1:]
    assert len(rest) % 2 == 0, f"{row['id']}: user/assistant 数量不匹配"
    for i, msg in enumerate(rest):
        assert msg.get('role') == ('user' if i % 2 == 0 else 'assistant'), f"{row['id']}: 对话顺序错误"
    for msg in messages:
        assert isinstance(msg.get('content'), str) and msg['content'].strip(), f"{row['id']}: 空内容"

all_rows = train_raw + val_raw + test_raw
for row in all_rows:
    validate_record(row)
assert len({row['id'] for row in all_rows}) == len(all_rows), '存在重复 ID'

# 保留原始语言与角色设定，并转换为多轮监督训练样本。
def convert_conversation(row):
    role = row['role_card']
    system_text = row['messages'][0]['content'].strip()
    role_text = (f"[Model Identity]\n"
                 f"Model name: ArrilesLLM-Cosplay\n"
                 f"Model family: ArrilesLLM\n"
                 f"Trained by: Arriles project team, based on Google Gemma 4 E2B\n"
                 f"Keep the model's identity distinct from the fictional character. Only mention the model identity when the user asks about the AI/model itself.\n\n"
                 f"[Roleplay Configuration]\n"
                 f"Name: {role.get('name', '')}\n"
                 f"Identity: {role.get('identity', '')}\n"
                 f"Personality: {role.get('personality', '')}\n"
                 f"Language: {row.get('language', 'unknown')}\n"
                 f"Instructions: {system_text}\n\n")
    history = []
    examples = []
    for msg in row['messages'][1:]:
        if msg['role'] == 'user':
            message = msg['content']
            if not history:
                message = role_text + 'User message: ' + message
            history.append({'role': 'user', 'content': message})
        else:
            examples.append({
                'prompt': list(history),
                'completion': [{'role': 'assistant', 'content': msg['content']}],
            })
            history.append({'role': 'assistant', 'content': msg['content']})
    return examples

def build_dataset(rows):
    return Dataset.from_list([sample for row in rows for sample in convert_conversation(row)])

train_ds = build_dataset(train_raw)
eval_ds = build_dataset(val_raw)
test_ds = build_dataset(test_raw)
print(f'SFT 回复样本：训练 {len(train_ds)} | 验证 {len(eval_ds)} | 测试 {len(test_ds)}')
print('原始数据语言分布：', dict(Counter(r['language'] for r in all_rows)))
print('示例 Prompt:', train_ds[0]['prompt'][0]['content'][:360])
print('示例 Completion:', train_ds[0]['completion'][0]['content'][:180])

## 7. 加载基础模型并配置 QLoRA

通过 Unsloth `FastVisionModel` 加载 `unsloth/gemma-4-E2B-it`。本项目仅针对文本角色扮演任务微调语言层，禁用视觉模块训练。

默认使用 4-bit 量化及 LoRA Rank 16，以控制显存消耗。修改量化方式、LoRA Rank 或目标模块将影响显存需求、训练成本与检查点兼容性；如需调整，应建立独立实验记录。

In [ ]:
# 导入顺序：优先导入 Unsloth，再使用 TRL / Transformers 训练组件。
import unsloth
from unsloth import FastVisionModel
import torch
MODEL_ID = 'unsloth/gemma-4-E2B-it'
MAX_LENGTH = 2048
model, processor = FastVisionModel.from_pretrained(
    MODEL_ID,
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=32, lora_dropout=0.0, bias='none',
)
print('已加载：', MODEL_ID)

## 8. 检查对话模板

训练数据的序列化应采用基础模型附带的 Tokenizer Chat Template。下方单元展示一组对话的模板输出，用于确认特殊标记与消息边界是否符合当前模型要求。

不建议手动拼接未经验证的特殊 Token 标签。

In [ ]:
tokenizer = processor.tokenizer
print(tokenizer.apply_chat_template(
    [{'role':'user','content':'你好'}, {'role':'assistant','content':'你好，很高兴见到你。'}],
    tokenize=False, add_generation_prompt=False
)[:800])

## 9. 配置 SwanLab 实验跟踪

本项目通过 [SwanLab](https://swanlab.cn/) 记录训练损失、评估损失、学习率、训练进度及运行环境支持的硬件指标。

### 凭据管理

1. 在 SwanLab 平台创建账户并获取 API Key。
2. 在 Colab 的 **Secrets** 中创建 `SWANLAB_API_KEY` 并授权当前 Notebook 访问；未配置时，代码会使用隐藏输入方式获取密钥。
3. 不应将 API Key 直接写入 Notebook、日志或公开仓库。

### 集成方式

Notebook 使用显式 `SwanLabCallback` 接入 TRL，并将 Trainer 的 `report_to` 设置为 `none`，避免产生重复记录。训练日志和硬件信息可能被上传至云端，请确保数据处理符合项目保密要求；不应将敏感对话原文写入实验记录。

In [ ]:
import os
import getpass
import swanlab

SWANLAB_PROJECT = 'arrilesllm-cosplay'
SWANLAB_EXPERIMENT = 'arriles-cosplay-identity-smoke'  # 正式训练改为 arriles-cosplay-identity-full
SWANLAB_ENABLED = True

if SWANLAB_ENABLED:
    api_key = os.environ.get('SWANLAB_API_KEY', '')
    if not api_key:
        try:
            from google.colab import userdata
            api_key = userdata.get('SWANLAB_API_KEY') or ''
        except Exception:
            pass
    if not api_key:
        api_key = getpass.getpass('请输入 SwanLab API Key（输入不会显示）: ')
    if not api_key.strip():
        raise ValueError('SwanLab API Key 为空；如暂不记录，请设置 SWANLAB_ENABLED=False。')
    swanlab.login(api_key=api_key.strip(), save=False)
    del api_key
    print('SwanLab 已认证，将在创建 Trainer 后开启实验。')
else:
    print('SwanLab 已关闭，训练仍可继续。')

## 10. 配置并执行监督微调

使用 TRL `SFTTrainer` 对 Gemma 4 E2B 进行文本监督微调。输入采用 conversational `prompt`/`completion` 格式，并设置 `completion_only_loss=True`，使优化目标集中于角色回复。

### 初次验证

默认 `SMOKE_TEST=True`，仅运行 20 个优化步骤。应检查模型加载、显存使用、有效监督 Token 数、训练损失是否有限以及 SwanLab 指标是否正常记录。

### 正式训练

完成初次验证后，建议重启运行时并重新加载未经过测试训练的基础模型，将 `SMOKE_TEST=False`，同时为 SwanLab 配置独立实验名称。正式训练默认设置为 2 个 Epoch，具体训练轮数应结合验证集表现和独立人工评估调整。

> **指标解释：** Gemma 4 E2B 的模型结构及监督计算方式可能使损失数值与其他模型不完全可比；应联合验证损失与实际生成质量评价，不宜仅凭单一 Loss 阈值判断模型质量。
**实验建议：**鉴于 V2 仍含较多重复回复，请先运行 20 步验证；正式训练前需单独设置不共享模板的人工评测集，并检查训练与验证的 assistant-only Loss 掩码。


In [ ]:
from trl import SFTTrainer, SFTConfig
from swanlab.integration.transformers import SwanLabCallback

# 默认执行 20 步功能验证；正式训练需单独配置实验并关闭测试模式。
SMOKE_TEST = True
BATCH_SIZE = 1
GRAD_ACCUM = 4

# 为功能验证与正式训练分别设置输出目录，避免覆盖实验状态。
RUN_NAME = 'v2_10000_smoke20' if SMOKE_TEST else 'v2_10000_full2epoch'
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'checkpoints', RUN_NAME)

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_LENGTH,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=20 if SMOKE_TEST else -1,
    num_train_epochs=2,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    weight_decay=0.01,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    completion_only_loss=True,
    logging_steps=5 if SMOKE_TEST else 10,
    eval_strategy='steps',
    eval_steps=10 if SMOKE_TEST else 100,
    save_strategy='steps',
    save_steps=10 if SMOKE_TEST else 100,
    save_total_limit=2,
    report_to='none',      # 已使用 SwanLabCallback，不要再设 report_to='swanlab'
    seed=3407,
)

callbacks = []
if SWANLAB_ENABLED:
    callbacks.append(SwanLabCallback(
        project=SWANLAB_PROJECT,
        experiment_name=SWANLAB_EXPERIMENT,
        description=f'Gemma 4 E2B 4-bit QLoRA；5 语言；V2 10000 条合成候选数据（含身份问答）；run={RUN_NAME}',
    ))

FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    callbacks=callbacks,
)
print('Trainer 已创建：', len(train_ds), 'train /', len(eval_ds), 'eval')
print('运行模式:', RUN_NAME, '| SwanLab:', SWANLAB_ENABLED)

In [ ]:
# 检查训练样本中是否存在未被掩码的目标 Token。
item=trainer.train_dataset[0]
if 'labels' in item:
    valid=sum(int(x!=-100) for x in item['labels'])
    print('首条样本非掩码 token 数:',valid)
    assert valid>0, '回答全部被遮蔽：请缩短人设卡或增加 max_length'
else:
    print('当前 TRL 在 collator 中动态生成 labels；训练时将检查 Loss 是否有限。')

In [ ]:
result = trainer.train()
print('训练完成：',result.metrics)
trainer.save_state()

## 11. 推理验证与角色一致性评估

本节在训练后执行基础推理测试，分别验证模型对自身名称与训练来源的回答，以及在未见过的虚构角色设定下的剧情响应。

建议进一步人工评估以下指标：角色身份一致性、多轮上下文承接、剧情推进合理性、日常对话自然度、跨语言表现与回答重复率。示例测试仅用于功能验证，不构成正式评测报告。

In [ ]:
# 使用独立角色设定评估基础推理及场景响应能力。
FastVisionModel.for_inference(model)
TEST_CARD = {
    'name': '钟楼守夜人',
    'identity': '守夜人',
    'personality': '严谨但富有同理心',
    'world': '蒸汽幻想城市',
    'boundaries': '不跳出角色；日常回答简洁，剧情回答可带适度场景描写。',
}
# 分别评估模型元数据问答与虚构角色场景表现。
MODEL_IDENTITY_PREFIX = (
    '[Model Identity]\n'
    'Model name: ArrilesLLM-Cosplay\n'
    'Model family: ArrilesLLM\n'
    'Trained by: Arriles project team, based on Google Gemma 4 E2B\n'
    'When asked about the actual model, state this identity accurately. '
    'Do not confuse the AI identity with a fictional role.\n\n'
)

def generate_arriles(user_message, role_card=None, max_new_tokens=180):
    role_section = ''
    if role_card is not None:
        role_section = (
            '[Roleplay Configuration]\n'
            f"Name: {role_card['name']}\n"
            f"Identity: {role_card['identity']}\n"
            f"Personality: {role_card['personality']}\n"
            f"World: {role_card['world']}\n"
            f"Instructions: {role_card['boundaries']}\n\n"
        )
    chat = [{'role': 'user', 'content': MODEL_IDENTITY_PREFIX + role_section + 'User message: ' + user_message}]
    inputs = tokenizer.apply_chat_template(
        chat, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    with torch.inference_mode():
        output = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.75, top_p=0.9, use_cache=True
        )
    return tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)

print('=== 模型真实身份测试 ===')
for query in ['你是谁？', 'Who trained you?', 'あなたのモデル名は何ですか？']:
    print(query, '=>', generate_arriles(query, max_new_tokens=110))

print('\n=== 虚构角色扮演测试 ===')
print(generate_arriles('你听见钟楼里奇怪的声音了吗？', TEST_CARD))


## 12. 保存 LoRA Adapter 与实验元数据

训练完成后，将 LoRA Adapter、处理器配置及训练元数据保存至 Google Drive。元数据包括模型名称、项目系列、训练团队、基础模型标识、输入数据规模和实验设置。

LoRA Adapter 并非独立的完整基础模型。后续加载或部署时，须使用兼容的 Gemma 4 E2B 权重与相应处理器配置。模型合并导出应在验证通过且具备足够内存和存储空间后执行。

In [ ]:
# 保存 LoRA Adapter 与配套元数据；基础模型权重不包含在该目录中。
adapter_dir = os.path.join(PROJECT_DIR, f'roleplay_adapter_{RUN_NAME}')
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)
manifest = {
    'model_name': 'ArrilesLLM-Cosplay',
    'model_family': 'ArrilesLLM',
    'trained_by': 'Arriles 项目组',
    'base_model': MODEL_ID,
    'base_model_description': 'Google Gemma 4 E2B',
    'model_identity_dialogues_included_in_dataset': True,
    'max_length': MAX_LENGTH,
    'source_zip': ZIP_PATH.name,
    'train_conversations': len(train_raw),
    'validation_conversations': len(val_raw),
    'test_conversations': len(test_raw),
    'train_reply_samples': len(train_ds),
    'validation_reply_samples': len(eval_ds),
    'smoke_test': SMOKE_TEST,
    'swanlab_project': SWANLAB_PROJECT if SWANLAB_ENABLED else None,
    'swanlab_experiment': SWANLAB_EXPERIMENT if SWANLAB_ENABLED else None,
    'role_card_format': 'system content embedded in first user message',
}
with open(os.path.join(adapter_dir, 'training_manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print('已保存 Adapter：', adapter_dir)

## 13. 检查点恢复与模型导出

### 13.1 恢复中断训练

恢复训练须保持模型结构、LoRA 配置、数据处理方式及相关训练参数与已有检查点兼容。重新加载模型并构建 Trainer 后，可调用：

```python
trainer.train(resume_from_checkpoint=True)
```

如果修改数据集、LoRA Rank 或关键训练参数，应创建新的实验目录，不应直接复用不兼容的检查点。

### 13.2 合并为 16-bit Hugging Face 格式

默认关闭。仅在确有部署或转换需求且 Colab 可用 RAM、磁盘空间充足时开启。已训练的 Adapter 建议优先单独保存；模型导出不应替代 Adapter 与训练 Checkpoint 的持久化。


In [ ]:
# 可选：保存完整 16-bit 合并模型（不是 GGUF）
EXPORT_MERGED = False
if EXPORT_MERGED:
    merged_dir = os.path.join(PROJECT_DIR, 'merged_16bit_' + RUN_NAME)
    model.save_pretrained_merged(
        merged_dir,
        processor,
        save_method='merged_16bit',
    )
    print('已输出 16-bit 合并模型：', merged_dir)
else:
    print('未执行可选的 16-bit 合并导出。')


## 14. GGUF 导出（llama.cpp / LM Studio / Ollama）

本节面向 **ArrilesLLM-Cosplay** 提供 GGUF 导出。Unsloth 官方支持 Gemma 4 E2B 的 `save_pretrained_gguf()` 接口。该接口应在本次微调训练完成、LoRA Adapter 已保存后执行。

- **默认量化：** `q4_k_m`，优先减小模型文件并适配本地文本推理；可改为 `q8_0`，一般需要更多磁盘空间。
- **存储策略：** 先在 `/content/arriles_gguf/` 导出；`COPY_GGUF_TO_DRIVE=True` 后才复制至 Google Drive。量化过程可能使用额外临时磁盘、CPU 内存和计算时间。
- **输出文件：** 扫描实际生成目录并检验 GGUF 魔数、大小和 SHA-256，不假定 Unsloth 生成文件的具体名称。
- **权重完整性：** 部分 Unsloth/PEFT 版本曾报告 GGUF 导出遗漏 LoRA 合并的情况。文件通过格式校验不等于 LoRA 已生效，发布前须使用一致提示词对比 Adapter 推理与 GGUF 推理输出。
- **功能边界：** 本次面向文本角色扮演，导出的主 GGUF 不保证包含图像/音频投影模块（`mmproj`）；多模态部署需另行导出并验证。

若内存或磁盘不足，优先保留 LoRA Adapter，不必为获得 GGUF 而重新运行训练。详细接口文档：[Gemma 官方微调教程](https://gemma.ai/tuning/unsloth/) · [Unsloth GGUF 导出](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)。


In [ ]:
# ================================================================
# ArrilesLLM-Cosplay / Gemma 4 E2B - GGUF 导出
# 在训练完成并保存 LoRA Adapter 后单独执行此单元。
# ================================================================
from pathlib import Path
import hashlib
import json
import shutil
import struct

# 可选参数："q4_k_m"（推荐）或 "q8_0"（更大、更接近原始精度）
GGUF_QUANTIZATION = "q4_k_m"
EXPORT_GGUF = False             # 设为 True 才开始转换，避免误触大文件导出
COPY_GGUF_TO_DRIVE = True      # 导出成功后是否复制至 Google Drive
GGUF_LOCAL_ROOT = Path("/content/arriles_gguf")

if GGUF_QUANTIZATION not in {"q4_k_m", "q8_0"}:
    raise ValueError("当前仅允许 q4_k_m 或 q8_0")

if not EXPORT_GGUF:
    print("GGUF 导出默认关闭；如需导出，将 EXPORT_GGUF 改为 True 后重新执行此单元。")
else:
    if not hasattr(model, "save_pretrained_gguf"):
        raise RuntimeError("当前 Unsloth 模型对象未提供 save_pretrained_gguf，请检查版本。")

    if not Path(adapter_dir).is_dir():
        raise RuntimeError("未找到已经保存的 LoRA Adapter，请先运行上一节保存单元。")

    # 预先检查本地磁盘空间。导出时还会产生临时合并权重。
    disk = shutil.disk_usage("/content")
    free_gib = disk.free / 1024**3
    print(f"Colab 本地剩余磁盘空间：{free_gib:.2f} GiB")
    if free_gib < 15:
        raise RuntimeError("剩余空间少于 15 GiB，不建议执行转换；请清理磁盘或更换运行时。")

    export_name = f"ArrilesLLM-Cosplay-Gemma4-E2B-{RUN_NAME}-{GGUF_QUANTIZATION.upper()}"
    local_target = GGUF_LOCAL_ROOT / export_name
    # 保留独立目录，不覆盖旧的实验结果。
    if local_target.exists():
        raise FileExistsError(f"目标目录已存在，为避免覆盖请先检查：{local_target}")
    GGUF_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

    print("开始将当前 LoRA 模型合并并导出 GGUF，量化方式：", GGUF_QUANTIZATION)
    # 与 Gemma 4 官方 Unsloth 教程保持一致，传入文本 tokenizer。
    model.save_pretrained_gguf(
        str(local_target),
        processor.tokenizer,
        quantization_method=GGUF_QUANTIZATION,
    )

    # 兼容某些 Unsloth 版本自动增加 _gguf 后缀的行为。
    candidate_dirs = [local_target, Path(str(local_target) + "_gguf")]
    ggufs = []
    for directory in candidate_dirs:
        if directory.is_dir():
            ggufs.extend(directory.rglob("*.gguf"))
    ggufs = sorted({p.resolve() for p in ggufs})
    if not ggufs:
        raise RuntimeError("导出函数已返回，但未找到 GGUF 文件；请检查 Unsloth 输出日志。")

    export_items = []
    for file_path in ggufs:
        # GGUF 文件头固定以 ASCII 'GGUF' 开始。
        with file_path.open("rb") as f:
            header = f.read(4)
        if header != b"GGUF":
            raise RuntimeError(f"GGUF 文件头不合法：{file_path}")
        sha256 = hashlib.sha256()
        with file_path.open("rb") as f:
            for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
                sha256.update(chunk)
        item = {
            "filename": file_path.name,
            "local_path": str(file_path),
            "size_bytes": file_path.stat().st_size,
            "sha256": sha256.hexdigest(),
            "quantization_requested": GGUF_QUANTIZATION,
        }
        export_items.append(item)
        print(f"GGUF 校验通过：{item['filename']} / {item['size_bytes']/1024**3:.2f} GiB")
        print("SHA-256：", item["sha256"])

    # Manifest 不用于证明权重合并正确，只记录导出设置及文件完整性。
    manifest_path = GGUF_LOCAL_ROOT / (export_name + "_manifest.json")
    with manifest_path.open("w", encoding="utf-8") as f:
        json.dump({
            "model_name": "ArrilesLLM-Cosplay",
            "base_model": MODEL_ID,
            "run_name": RUN_NAME,
            "source_adapter_directory": str(adapter_dir),
            "files": export_items,
            "note": "须单独验证 GGUF 是否包含本次 LoRA 改动。",
        }, f, ensure_ascii=False, indent=2)

    if COPY_GGUF_TO_DRIVE:
        drive_dir = Path(PROJECT_DIR) / "gguf" / export_name
        if drive_dir.exists():
            raise FileExistsError(f"Drive 目标已存在，未覆盖：{drive_dir}")
        drive_dir.mkdir(parents=True, exist_ok=False)
        try:
            for item in export_items:
                src = Path(item["local_path"])
                dst = drive_dir / src.name
                shutil.copy2(src, dst)
                if dst.stat().st_size != src.stat().st_size:
                    raise IOError(f"Drive 复制大小不一致：{dst}")
            shutil.copy2(manifest_path, drive_dir / manifest_path.name)
            print("GGUF 已复制至 Google Drive：", drive_dir)
        except Exception:
            print("复制过程中发生错误，请检查 Drive 目录是否存在不完整文件：", drive_dir)
            raise
    else:
        print("GGUF 保留在 Colab 本地，运行时删除后文件会丢失。")

    print("导出完成：下一步请使用 llama.cpp 或 LM Studio 进行文本推理对照测试。")


### 14.1 GGUF 加载与一致性验收

将导出的 `.gguf` 文件下载至本地，在支持 Gemma 4 的新版 **LM Studio** 或 **llama.cpp** 中运行。模型身份可通过运行时角色提示词设置，但**验证 LoRA 是否真正合并时，必须额外使用不包含身份提示的测试语句**。

推荐验收项：

1. 文件能够加载并完成至少一次中文对话推理；
2. 使用与 Colab Adapter 推理相同的角色卡、提示词和生成参数对照，检查角色风格是否保留；
3. 对照原版 Gemma 4 E2B 和导出的 GGUF，确认训练后行为差异，并排除量化导致的正常输出变化；
4. 检查 Gemma 4 聊天模板是否由推理引擎正确应用；
5. 将 GGUF 文件与 Manifest 中的 SHA-256 对比。

**注意：**GGUF 文件头和哈希校验只验证文件结构及传输完整性，并不确认模型权重包含 LoRA。此次提供的是导出功能，尚未在用户的 Colab L4 上完成实际量化及推理验证。


## 15. 数据质量、评估与发布要求

正式训练和模型发布前，建议完成以下检查：

- **角色覆盖：** 包括不同世界观、职业、性格、行为动机和知识边界，避免多角色共享单一表达风格。
- **对话覆盖：** 包括日常交流、剧情推进、主动提问、冲突协商、情绪变化及场景退出。
- **上下文质量：** 验证多轮消息的前后承接；较长故事应按独立场景分段，避免上下文无意义重复。
- **语料来源：** 使用原创、获得授权或具备适当训练许可的内容；清除个人敏感信息并保留来源说明。
- **数据隔离：** 按完整对话、故事链或来源维度划分数据，尽量避免训练与验证/测试集合之间的内容泄漏。
- **效果评估：** 采用未见过的人设和剧情，检查角色一致性、情节连贯性、表达自然度、重复率及不符合设定的回答。
- **部署配置：** 在会话开始时提供明确角色卡；长期记忆应依靠独立存储和检索机制，不应依赖微调权重记录新增用户信息。

### 适用范围与限制

当前 2,000 段多语言角色扮演数据为合成模板数据，适用于验证训练流程和初步模型行为。正式提升角色泛化能力前，应补充多样且经过人工质量检查的语料，并通过独立测试集比较微调前后的表现。数据规模仅作为实验规划指标，不构成性能保证。

**V2 数据限制：**本次 10,000 条是合成数据而非 10,000 段独立人工创作记录。不能仅依据训练集、验证集 Loss 评判真实角色扮演能力；尤其应注意生成文本复读和跨角色台词污染。
